# Exploratory Data Analysis  - CNEFE Addresses

## Read Data

In [2]:
from pyspark.sql import SparkSession
from pathlib import Path
import os


current_path = os.getcwd()
project_root = Path(current_path).parent
os.chdir(project_root)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
spark = SparkSession.builder \
    .appName("Read Integrated Data") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
df = spark.read.parquet("data/integrated/integrated_cnefe_addresses.parquet")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/20 22:51:17 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/07/20 22:51:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/20 22:51:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/20

In [3]:
df.count()


111102875

## analyse missingness

In [4]:
from pyspark.sql import functions as F

total_rows = df.count()

fulfillment_exprs = [
    F.struct(
        F.lit(c).alias("column"),
        (F.count(F.col(c)) / F.lit(total_rows) * 100).alias("fulfillment_pct")
    )
    for c in df.columns
]

result = df.select(F.array(*fulfillment_exprs).alias("stats")) \
    .selectExpr("explode(stats) as stats") \
    .select("stats.column", "stats.fulfillment_pct")

result.orderBy(F.col("fulfillment_pct").desc()).show(df.columns.__len__(), truncate=False)

26/07/20 22:51:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------+--------------------+
|column                        |fulfillment_pct     |
+------------------------------+--------------------+
|COD_UNICO_ENDERECO            |100.0               |
|COD_UF                        |100.0               |
|COD_MUNICIPIO                 |100.0               |
|COD_DISTRITO                  |100.0               |
|COD_SUBDISTRITO               |100.0               |
|COD_SETOR                     |100.0               |
|NUM_QUADRA                    |100.0               |
|NUM_FACE                      |100.0               |
|CEP                           |100.0               |
|NOM_TIPO_SEGLOGR              |100.0               |
|NOM_SEGLOGR                   |100.0               |
|NUM_ENDERECO                  |100.0               |
|LATITUDE                      |100.0               |
|LONGITUDE                     |100.0               |
|NV_GEO_COORD                  |100.0               |
|COD_ESPECIE                

## análise Contagem valores dos campos

In [5]:
for c in df.columns:
    print(f"Column: {c}")
    df.filter(F.col(c).isNotNull()).groupBy(c).count().orderBy(F.col("count").desc()).show(100, truncate=False)

Column: COD_UNICO_ENDERECO


+------------------+-----+
|COD_UNICO_ENDERECO|count|
+------------------+-----+
|16360870          |6    |
|35577467          |5    |
|211407776         |5    |
|223042492         |5    |
|20261400          |4    |
|25170284          |4    |
|25540858          |4    |
|26667198          |4    |
|27616396          |4    |
|47152777          |4    |
|51768699          |4    |
|67684803          |4    |
|81294391          |4    |
|110889910         |4    |
|110896336         |4    |
|111334461         |4    |
|111498242         |4    |
|111498329         |4    |
|113337495         |4    |
|192541788         |4    |
|206160952         |4    |
|224590634         |4    |
|33168411          |4    |
|47952734          |4    |
|49455699          |4    |
|57701167          |4    |
|58733606          |4    |
|80849539          |4    |
|95990777          |4    |
|101624147         |4    |
|110740666         |4    |
|112939435         |4    |
|186971202         |4    |
|209643509         |4    |
|

+------+--------+
|COD_UF|count   |
+------+--------+
|35    |22953725|
|31    |12038190|
|29    |9047296 |
|33    |8962200 |
|43    |6456747 |
|41    |6122025 |
|26    |5120482 |
|23    |4750642 |
|42    |4181139 |
|52    |3960937 |
|15    |3911170 |
|21    |3257843 |
|25    |2228886 |
|32    |2221348 |
|51    |1985314 |
|22    |1891421 |
|24    |1845003 |
|13    |1658970 |
|27    |1626565 |
|50    |1507819 |
|53    |1318887 |
|28    |1258755 |
|11    |965370  |
|17    |848437  |
|12    |410524  |
|16    |312665  |
|14    |260515  |
+------+--------+

Column: COD_MUNICIPIO
+-------------+-------+
|COD_MUNICIPIO|count  |
+-------------+-------+
|3550308      |5689391|
|3304557      |3276028|
|2927408      |1390897|
|5300108      |1318887|
|2304400      |1193529|
|3106200      |1180102|
|4106902      |900459 |
|1302603      |878488 |
|5208707      |777018 |
|4314902      |762239 |
|2611606      |737511 |
|1501402      |618075 |
|3518800      |596864 |
|3509502      |574086 |
|2111300   

+------------+-------+
|COD_DISTRITO|count  |
+------------+-------+
|330455705   |3276028|
|292740805   |1390897|
|530010805   |1318887|
|230440005   |1193529|
|410690205   |900459 |
|130260305   |878488 |
|520870705   |775706 |
|431490205   |762239 |
|261160605   |737511 |
|310620005   |726983 |
|350950205   |515772 |
|211130005   |481179 |
|270430205   |466514 |
|500270405   |457559 |
|250750705   |426733 |
|330350005   |416521 |
|221100105   |415723 |
|240810205   |395998 |
|351880005   |387294 |
|317020605   |377919 |
|354340205   |360289 |
|354870805   |359628 |
|355220505   |355399 |
|353440105   |328129 |
|310620060   |319313 |
|354780905   |318929 |
|291080005   |315081 |
|280030805   |307745 |
|313670205   |300155 |
|354990405   |278596 |
|411370005   |272949 |
|420910205   |269751 |
|330330205   |254466 |
|430510805   |240187 |
|354980505   |237406 |
|330045605   |233172 |
|354850005   |231082 |
|150080005   |220206 |
|110020505   |217899 |
|311860105   |209850 |
|310670505 

+---------------+------+
|COD_SUBDISTRITO|count |
+---------------+------+
|43149020500    |762239|
|35095020500    |515772|
|21113000500    |481179|
|25075070500    |426733|
|35188000500    |387294|
|31702060500    |377919|
|35434020500    |360289|
|35487080500    |359628|
|35522050500    |355399|
|33045570521    |333230|
|35344010500    |328129|
|35478090500    |318929|
|28003080500    |307745|
|33045570523    |305095|
|35499040500    |278596|
|42091020500    |269751|
|33033020500    |254466|
|35498050500    |237406|
|35485000500    |231082|
|13026030512    |227665|
|29274080522    |224957|
|15008000500    |220206|
|33045570524    |219994|
|33045570529    |217627|
|35188001000    |209570|
|31433020500    |209199|
|29274080518    |208721|
|35259040500    |207882|
|35410000500    |206098|
|52011080500    |203530|
|25040090500    |202469|
|26041060500    |201903|
|35060030500    |199387|
|35294010500    |195094|
|33045570517    |192472|
|33045570522    |191472|
|26116060523    |188308|


...

+----------+-----+
|LONGITUDE |count|
+----------+-----+
|-48.919712|1011 |
|-49.285104|884  |
|-46.771631|842  |
|-48.630703|778  |
|-43.299967|763  |
|-48.636012|718  |
|-38.493439|668  |
|-46.869385|658  |
|-46.577765|644  |
|-48.449551|633  |
|-46.780648|629  |
|-38.411958|615  |
|-38.477901|613  |
|-48.014896|612  |
|-56.092535|611  |
|-49.262218|605  |
|-38.938995|596  |
|-51.166153|585  |
|-56.070087|584  |
|-46.71801 |582  |
|-51.140757|582  |
|-51.149382|582  |
|-46.711459|580  |
|-43.341258|579  |
|-38.27261 |572  |
|-47.420249|568  |
|-46.614423|567  |
|-38.534125|566  |
|-34.907105|560  |
|-51.190671|560  |
|-43.253808|556  |
|-43.348514|555  |
|-46.433876|550  |
|-49.28577 |549  |
|-48.96653 |547  |
|-48.430815|541  |
|-38.561402|539  |
|-49.161181|538  |
|-46.869371|537  |
|-43.309944|535  |
|-46.449917|527  |
|-48.448495|525  |
|-48.448557|524  |
|-34.862486|524  |
|-49.347413|522  |
|-46.691822|520  |
|-46.536819|518  |
|-48.660753|518  |
|-34.945771|512  |
|-46.816919|

+-----------+--------+
|COD_ESPECIE|count   |
+-----------+--------+
|1          |90600065|
|6          |11706713|
|3          |4058385 |
|7          |3541442 |
|8          |579798  |
|4          |264445  |
|5          |247510  |
|2          |104517  |
+-----------+--------+

Column: DSC_ESTABELECIMENTO


+-------------------------------+------+
|DSC_ESTABELECIMENTO            |count |
+-------------------------------+------+
|VAGO                           |306558|
|BAR                            |285895|
|DEPOSITO                       |239801|
|GARAGEM                        |232729|
|SEM NOME                       |120131|
|GALPAO                         |107896|
|SEM DENOMINACAO                |105884|
|IGREJA                         |82349 |
|SALAO DE BELEZA                |80926 |
|SITIO                          |74906 |
|CRIACAO DE GADO                |72132 |
|OFICINA                        |71979 |
|CURRAL                         |70727 |
|LOJA VAGA                      |70269 |
|MERCEARIA                      |69576 |
|LOJA                           |57994 |
|OFICINA MECANICA               |57012 |
|CASA DE FARINHA                |51357 |
|FAZENDA                        |49116 |
|BARBEARIA                      |46520 |
|LANCHONETE                     |44972 |
|BARRACAO       

+---------------+--------+
|COD_TIPO_ESPECI|count   |
+---------------+--------+
|101            |73817837|
|103            |13942183|
|102            |2442534 |
|104            |502028  |
+---------------+--------+

Column: UF


+---+--------+
|UF |count   |
+---+--------+
|SP |22953725|
|MG |12038190|
|BA |9047296 |
|RJ |8962200 |
|RS |6456747 |
|PR |6122025 |
|PE |5120482 |
|CE |4750642 |
|SC |4181139 |
|GO |3960937 |
|PA |3911170 |
|MA |3257843 |
|PB |2228886 |
|ES |2221348 |
|MT |1985314 |
|PI |1891421 |
|RN |1845003 |
|AM |1658970 |
|AL |1626565 |
|MS |1507819 |
|DF |1318887 |
|SE |1258755 |
|RO |965370  |
|TO |848437  |
|AC |410524  |
|AP |312665  |
|RR |260515  |
+---+--------+



## Look if null values are consistent

### Based on empty strings

In [6]:
from pyspark.sql import functions as F

string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]

empty_string_exprs = [
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols
]

df.select(empty_string_exprs).show(vertical=True, truncate=False)

-RECORD 0-----------------------------
 COD_SETOR                      | 0   
 CEP                            | 0   
 DSC_LOCALIDADE                 | 0   
 NOM_TIPO_SEGLOGR               | 0   
 NOM_TITULO_SEGLOGR             | 0   
 NOM_SEGLOGR                    | 0   
 DSC_MODIFICADOR                | 0   
 NOM_COMP_ELEM1                 | 0   
 VAL_COMP_ELEM1                 | 0   
 NOM_COMP_ELEM2                 | 0   
 VAL_COMP_ELEM2                 | 0   
 NOM_COMP_ELEM3                 | 0   
 VAL_COMP_ELEM3                 | 0   
 NOM_COMP_ELEM4                 | 0   
 VAL_COMP_ELEM4                 | 0   
 NOM_COMP_ELEM5                 | 0   
 VAL_COMP_ELEM5                 | 0   
 DSC_ESTABELECIMENTO            | 0   
 COD_INDICADOR_ESTAB_ENDERECO   | 0   
 COD_INDICADOR_CONST_ENDERECO   | 0   
 COD_INDICADOR_FINALIDADE_CONST | 0   
 UF                             | 0   



### Based on String which sings value is undefined

In [7]:
from functools import reduce

UNDEFINED_EXPRESSIONS = [
    "SEM DENOMINACAO",
    "VAGO"
]

total_rows = df.count()

string_cols = {f.name for f in df.schema.fields if str(f.dataType) == "StringType()"}

def is_undefined(col):
    return reduce(
        lambda a, b: a | b,
        [col.contains(exp) for exp in UNDEFINED_EXPRESSIONS]
    )

fulfillment_exprs = []
for c in df.columns:
    if c in string_cols:
        condition = F.col(c).isNotNull() & (~is_undefined(F.col(c)))
    else:
        condition = F.col(c).isNotNull()

    fulfillment_exprs.append(
        F.struct(
            F.lit(c).alias("column"),
            (F.count(F.when(condition, F.lit(1))) / F.lit(total_rows) * 100).alias("fulfillment_pct")
        )
    )

result = df.select(F.array(*fulfillment_exprs).alias("stats")) \
    .selectExpr("explode(stats) as stats") \
    .select("stats.column", "stats.fulfillment_pct")

result.orderBy(F.col("fulfillment_pct").desc()).show(df.columns.__len__(), truncate=False)

+------------------------------+--------------------+
|column                        |fulfillment_pct     |
+------------------------------+--------------------+
|COD_UNICO_ENDERECO            |100.0               |
|COD_UF                        |100.0               |
|COD_MUNICIPIO                 |100.0               |
|COD_DISTRITO                  |100.0               |
|COD_SUBDISTRITO               |100.0               |
|COD_SETOR                     |100.0               |
|NUM_QUADRA                    |100.0               |
|NUM_FACE                      |100.0               |
|CEP                           |100.0               |
|NOM_TIPO_SEGLOGR              |100.0               |
|NUM_ENDERECO                  |100.0               |
|LATITUDE                      |100.0               |
|LONGITUDE                     |100.0               |
|NV_GEO_COORD                  |100.0               |
|COD_ESPECIE                   |100.0               |
|UF                         

In [8]:
from pyspark.sql import functions as F
from functools import reduce

UNDEFINED_EXPRESSIONS = [
    "SEM DENOMINACAO",
    "VAGO"
]

total_rows = df.count()

string_cols = {f.name for f in df.schema.fields if str(f.dataType) == "StringType()"}

def is_undefined(col):
    return reduce(
        lambda a, b: a | b,
        [col.contains(exp) for exp in UNDEFINED_EXPRESSIONS]
    )

# Build both conditions (sem filtro / com filtro) in a single pass
exprs = []
for c in df.columns:
    # Sem filtro: apenas remove nulos
    not_null_cond = F.col(c).isNotNull()

    # Com filtro: remove nulos + expressões indefinidas (apenas colunas string)
    if c in string_cols:
        full_cond = F.col(c).isNotNull() & (~is_undefined(F.col(c)))
    else:
        full_cond = not_null_cond

    exprs.append(
        F.struct(
            F.lit(c).alias("column"),
            (F.count(F.when(not_null_cond, F.lit(1))) / F.lit(total_rows) * 100).alias("pct_filtro_null_basico"),
            (F.count(F.when(full_cond, F.lit(1))) / F.lit(total_rows) * 100).alias("pct_com_filtro_null_strito")
        )
    )

result = df.select(F.array(*exprs).alias("stats")) \
    .selectExpr("explode(stats) as stats") \
    .select(
        "stats.column",
        "stats.pct_filtro_null_basico",
        "stats.pct_com_filtro_null_strito"
    ) \
    .withColumn("diferenca", F.round(F.col("pct_com_filtro_null_strito") - F.col("pct_filtro_null_basico"), 4)) \
    .withColumn("pct_filtro_null_basico", F.round(F.col("pct_filtro_null_basico"), 4)) \
    .withColumn("pct_com_filtro_null_strito", F.round(F.col("pct_com_filtro_null_strito"), 4))

result.orderBy(F.col("pct_com_filtro_null_strito").desc()).show(df.columns.__len__(), truncate=False)

+------------------------------+----------------------+--------------------------+---------+
|column                        |pct_filtro_null_basico|pct_com_filtro_null_strito|diferenca|
+------------------------------+----------------------+--------------------------+---------+
|COD_UNICO_ENDERECO            |100.0                 |100.0                     |0.0      |
|COD_UF                        |100.0                 |100.0                     |0.0      |
|COD_MUNICIPIO                 |100.0                 |100.0                     |0.0      |
|COD_DISTRITO                  |100.0                 |100.0                     |0.0      |
|COD_SUBDISTRITO               |100.0                 |100.0                     |0.0      |
|COD_SETOR                     |100.0                 |100.0                     |0.0      |
|NUM_QUADRA                    |100.0                 |100.0                     |0.0      |
|NUM_FACE                      |100.0                 |100.0          

#### Plot differences

In [9]:
from pyspark.sql import functions as F
import plotly.graph_objects as go

diff_df = result.filter(F.col("diferenca") != 0.0) \
            .orderBy(F.col("diferenca").asc())
    
diff_rows = diff_df.select("column", "diferenca").collect()

labels = [row["column"] for row in diff_rows]
data = [round(row["diferenca"], 4) for row in diff_rows]

fig = go.Figure(
    go.Bar(
        x=data,
        y=labels,
        orientation="h",
        marker=dict(color="red"),
        text=[f"{v:.4f}" for v in data],
        textposition="outside",
        hovertemplate="%{y}: %{x:.4f} p.p.<extra></extra>"
    )
)

fig.update_layout(
    title="Diferença de fulfillment (com filtro mais restritivo - filtro nulo básico)",
    xaxis_title="Diferença (p.p.)",
    yaxis_title="Coluna",
    yaxis=dict(autorange="reversed"),  # mantém a maior diferença no topo
    template="plotly_white",
    height=max(400, len(labels) * 40 + 100)
)



## Assay of Cleaning Data

Options of cleaning Data we may consider:

- Trim leading and trailing whitespace from string values (e.g., " john smith " becomes "john smith").
- Remove special characters from string values (e.g., "O'Hara" becomes "Ohara").
- Standardise date formats as strings in "yyyy-mm-dd" format.
- Replace abbreviations with full words (e.g., standardize "St." and "Street" to "Street").